# Multi-Telescope Color Composite Imaging
## Combining NOT and LT images within 1 day

This notebook:
1. Loads FITS files from NOT and LT telescopes
2. Extracts object names, metadata, and seeing
3. Groups NOT+LT images by temporal proximity (within 1 day)
4. Measures seeing for NOT images (uses L1SEESEC header keyword for LT)
5. Creates color composite images using WCS alignment

In [1]:
import warnings
warnings.filterwarnings('ignore')

from glob import glob
import pandas as pd
import numpy as np
from astropy.io import fits
from astropy.time import Time
import matplotlib.pyplot as plt
from pathlib import Path

# Import our custom modules
from seeing_measure import measure_seeing_batch, seeing_summary
from color_image_notebook import ColorImageMaker

%matplotlib inline

## Step 1: Load Data and Extract Metadata

In [2]:
# Collect all FITS files
not_files = glob('/Users/kryanhinds/subphot_pipe/data/NOT_imaging/*/*.fits') + \
            glob('/Users/kryanhinds/subphot_pipe/data/NOT_20260423-24/*.fits')
tjo_files = glob('/Users/kryanhinds/subphot_pipe/data/TJO/*.fits')
lt_files  = glob('/Users/kryanhinds/subphot_pipe/data/ZTF26aakjzdt_LT/*/*.fits')

all_files = not_files + tjo_files + lt_files

print(f"Found {len(not_files)} NOT files")
print(f"Found {len(tjo_files)} TJO files")
print(f"Found {len(lt_files)} LT files")
print(f"Total: {len(all_files)} files")

Found 21 NOT files
Found 15 TJO files
Found 61 LT files
Total: 97 files


In [3]:
# Extract filter, MJD, and instrument from each file
not_kw = {'filt':'FAFLTNM', 'obs_date':'DATE-OBS', 'inst':'NOT/ALFOSC', 'exp_time':'EXPTIME'}
tjo_kw = {'filt':'FILTER', 'obs_date':['DATE-OBS','TIME-OBS'], 'inst':'TJO/MEIA2', 'exp_time':'EXPTIME'}
lt_kw  = {'filt':'FILTER1', 'obs_date':'DATE-OBS', 'inst':'LT/IOO', 'exp_time':'EXPTIME', 'see':'L1SEESEC'}

data = []
for f in all_files:
    try:
        with fits.open(f) as hdul:
            hdr = hdul[0].header
            
            if 'NOT' in f:
                filt = hdr[not_kw['filt']][0]
                obs_date = hdr[not_kw['obs_date']]
                exp_time = hdr[not_kw['exp_time']]
                inst = not_kw['inst']
                obj = hdr.get('OBJECT', 'UNKNOWN').strip()
                seeing = None  # Will be measured
                
            elif 'TJO' in f:
                filt = hdr[tjo_kw['filt']][-1]
                obs_date = Time(hdr[tjo_kw['obs_date'][0]] + 'T' + hdr[tjo_kw['obs_date'][1]]).iso
                exp_time = hdr[tjo_kw['exp_time']]
                inst = tjo_kw['inst']
                obj = hdr.get('OBJECT', 'UNKNOWN').strip()
                seeing = None  # Will be measured
                
            elif 'LT' in f:
                filt = hdr[lt_kw['filt']][-1].lower()
                obs_date = hdr[lt_kw['obs_date']]
                exp_time = hdr[lt_kw['exp_time']]
                inst = lt_kw['inst']
                obj = hdr.get('OBJECT', 'UNKNOWN').strip()
                # Use L1SEESEC from header for LT images
                seeing = hdr.get(lt_kw['see'], None)
            else:
                continue
            
            obs_mjd = Time(obs_date).mjd + (exp_time/2) / 86400.0
            
            data.append({
                'filename': f,
                'instrument': inst,
                'filter': filt.lower(),
                'obs_date': obs_date,
                'obs_mjd': obs_mjd,
                'exp_time': exp_time,
                'object': obj,
                'seeing': seeing
            })
    except Exception as e:
        print(f"Error reading {Path(f).name}: {e}")

df = pd.DataFrame(data)
print(f"\nSuccessfully loaded {len(df)} images")


Successfully loaded 97 images


In [4]:
# Display summary
print("Data summary:")
print(f"\nShape: {df.shape}")
print(f"\nInstruments:")
print(df['instrument'].value_counts())
print(f"\nFilters:")
print(df['filter'].value_counts())
print(f"\nObjects:")
print(df['object'].value_counts())
print(f"\nMJD range: {df['obs_mjd'].min():.3f} - {df['obs_mjd'].max():.3f}")
print(f"\nSeeing info:")
print(f"  LT images with seeing: {df[(df['instrument'] == 'LT/IOO') & (df['seeing'].notna())].shape[0]} / {len(df[df['instrument'] == 'LT/IOO'])}")
if len(df[(df['instrument'] == 'LT/IOO') & (df['seeing'].notna())]) > 0:
    lt_seeing = df[(df['instrument'] == 'LT/IOO') & (df['seeing'].notna())]['seeing']
    print(f"  LT seeing range: {lt_seeing.min():.2f}\" - {lt_seeing.max():.2f}\"")

Data summary:

Shape: (97, 8)

Instruments:
instrument
LT/IOO        61
NOT/ALFOSC    21
TJO/MEIA2     15
Name: count, dtype: int64

Filters:
filter
g    30
r    28
i    27
z    10
u     2
Name: count, dtype: int64

Objects:
object
ZTF26aakjzdt    49
GRB260310A      32
AT 2026fgk      15
pg1633           1
Name: count, dtype: int64

MJD range: 61112.113 - 61156.168

Seeing info:
  LT images with seeing: 61 / 61
  LT seeing range: 1.04" - 2.66"


## Step 2: Find Temporal Groups (NOT + LT within 1 day)

In [5]:
def find_temporal_groups_mjd(df, time_window=1.0, instruments=None):
    """
    Group observations by MJD proximity across instruments.
    Doesn't require same object name - just temporal proximity.
    """
    if instruments:
        df_filtered = df[df['instrument'].isin(instruments)].copy()
    else:
        df_filtered = df.copy()
    
    df_filtered = df_filtered.sort_values('obs_mjd').reset_index(drop=True)
    groups = []
    used = set()
    
    for i, row in df_filtered.iterrows():
        if i in used:
            continue
        
        mjd_center = row['obs_mjd']
        # Find all images within time_window
        mask = (df_filtered['obs_mjd'] >= mjd_center - time_window) & \
               (df_filtered['obs_mjd'] <= mjd_center + time_window)
        group = df_filtered[mask].copy()
        
        # Only keep if we have multiple instruments
        if len(group['instrument'].unique()) > 1:
            groups.append(group.reset_index(drop=True))
            used.update(group.index)
    
    return groups

# Find temporal groups: NOT + LT only, within 1 day
temporal_groups = find_temporal_groups_mjd(
    df,
    time_window=1.0,
    instruments=['NOT/ALFOSC', 'LT/IOO']
)

print(f"Found {len(temporal_groups)} temporal groups (NOT + LT within 1 day)\n")

Found 2 temporal groups (NOT + LT within 1 day)



In [6]:
# Display each group
for group_id, group in enumerate(temporal_groups):
    print(f"\n{'='*70}")
    print(f"Group {group_id + 1}")
    print(f"{'='*70}")
    print(f"MJD span: {group['obs_mjd'].max() - group['obs_mjd'].min():.4f} days")
    print(f"MJD range: {group['obs_mjd'].min():.3f} - {group['obs_mjd'].max():.3f}")
    print(f"Instruments: {', '.join(sorted(group['instrument'].unique()))}")
    print(f"Filters: {', '.join(sorted(group['filter'].unique()))}")
    print(f"Total images: {len(group)}")
    print(f"\nBreakdown by instrument:")
    for inst in sorted(group['instrument'].unique()):
        inst_group = group[group['instrument'] == inst]
        seeing_info = inst_group['seeing'].dropna()
        print(f"  {inst}: {len(inst_group)} images, filters: {','.join(sorted(inst_group['filter'].unique()))}")
        if len(seeing_info) > 0:
            print(f"    Seeing: {seeing_info.min():.2f}\" - {seeing_info.max():.2f}\" (median: {seeing_info.median():.2f}\")")


Group 1
MJD span: 0.9980 days
MJD range: 61114.991 - 61115.989
Instruments: LT/IOO, NOT/ALFOSC
Filters: g, i, r, z
Total images: 16

Breakdown by instrument:
  LT/IOO: 12 images, filters: g,i,r,z
    Seeing: 1.69" - 2.12" (median: 1.82")
  NOT/ALFOSC: 4 images, filters: g,i,r,z

Group 2
MJD span: 0.0519 days
MJD range: 61154.045 - 61154.097
Instruments: LT/IOO, NOT/ALFOSC
Filters: g, i, r
Total images: 16

Breakdown by instrument:
  LT/IOO: 13 images, filters: g,i,r
    Seeing: 1.18" - 1.66" (median: 1.45")
  NOT/ALFOSC: 3 images, filters: g,i,r


## Step 3: Measure Seeing (for NOT images only; LT uses L1SEESEC)

In [7]:
# Measure seeing only for NOT and TJO images (LT already has L1SEESEC)
print("Measuring seeing for NOT images...\n")

# Only measure for NOT images that don't have seeing yet
not_mask = (df['instrument'] == 'NOT/ALFOSC') & (df['seeing'].isna())
if not_mask.sum() > 0:
    df.loc[not_mask, 'seeing'] = measure_seeing_batch(
        df.loc[not_mask, 'filename'],
        df.loc[not_mask, 'instrument'],
        threshold=2.5,
        verbose=True
    )

print(f"\nSeeing measurements complete")
print(f"Total images with seeing: {df['seeing'].notna().sum()} / {len(df)}")

Measuring seeing for NOT images...

[1/21] GRB260310A_astro_g.fits...   FWHM: 2.64 pix → 0.50 arcsec
✓ 0.50"
[2/21] GRB260310A_astro_r.fits...   FWHM: 2.49 pix → 0.47 arcsec
✓ 0.47"
[3/21] GRB260310A_astro_z.fits...   FWHM: 1.85 pix → 0.35 arcsec
✓ 0.35"
[4/21] GRB260310A_astro_i.fits...   FWHM: 5.51 pix → 1.04 arcsec
✓ 1.04"
[5/21] GRB260310A_astro_g.fits...   FWHM: 0.79 pix → 0.15 arcsec
✓ 0.15"
[6/21] GRB260310A_astro_r.fits...   FWHM: 0.77 pix → 0.15 arcsec
✓ 0.15"
[7/21] GRB260310A_astro_z.fits...   FWHM: 4.33 pix → 0.82 arcsec
✓ 0.82"
[8/21] GRB260310A_astro_i.fits...   FWHM: 4.66 pix → 0.88 arcsec
✓ 0.88"
[9/21] GRB260310A_astro_g.fits...   FWHM: 4.22 pix → 0.80 arcsec
✓ 0.80"
[10/21] GRB260310A_astro_r.fits...   FWHM: 4.62 pix → 0.87 arcsec
✓ 0.87"
[11/21] GRB260310A_astro_z.fits...   FWHM: 9.75 pix → 1.84 arcsec
✓ 1.84"
[12/21] GRB260310A_astro_i.fits...   FWHM: 2.90 pix → 0.55 arcsec
✓ 0.55"
[13/21] GRB260310A_astro_g.fits...   FWHM: 4.77 pix → 0.90 arcsec
✓ 0.90"
[14/21] GRB

In [8]:
# Show seeing statistics by instrument
print("Seeing Statistics by Instrument:\n")

for inst in sorted(df['instrument'].unique()):
    inst_df = df[df['instrument'] == inst]
    seeing_vals = inst_df['seeing'].dropna()
    
    if len(seeing_vals) > 0:
        print(f"{inst}:")
        print(f"  Count: {len(seeing_vals)}")
        print(f"  Median: {seeing_vals.median():.2f}\"")
        print(f"  Mean:   {seeing_vals.mean():.2f}\"")
        print(f"  Range:  {seeing_vals.min():.2f}\" - {seeing_vals.max():.2f}\"")
        print()

Seeing Statistics by Instrument:

LT/IOO:
  Count: 61
  Median: 1.58"
  Mean:   1.59"
  Range:  1.04" - 2.66"

NOT/ALFOSC:
  Count: 21
  Median: 0.55"
  Mean:   0.67"
  Range:  0.15" - 1.84"



## Step 4: Select Best Images and Create Color Composites

In [9]:
# Function to create color image from a temporal group
def create_color_from_group(group, seeing_limit=2.0, output_name=None, verbose=True):
    """
    Create color image from a temporal group.
    
    Parameters
    ----------
    group : pd.DataFrame
        Temporal group from temporal_groups
    seeing_limit : float
        Maximum seeing in arcseconds
    output_name : str, optional"
        Output filename (e.g., 'color_group_0.png')
    verbose : bool
        Print progress
    
    Returns
    -------
    (rgb_image, maker, info) or None
        RGB image, ColorImageMaker object, and info dict. None if creation failed.
    """
    # Filter by seeing
    good = group[(group['seeing'] < seeing_limit) & (group['seeing'].notna())]
    
    if verbose:
        print(f"After seeing filter (<{seeing_limit}\"): {len(good)} / {len(group)} images")
    
    if len(good) < 3:
        if verbose:
            print(f"Not enough good images for color composite")
        return None
    
    # Check we have multiple filters
    if len(good['filter'].unique()) < 3:
        if verbose:
            print(f"Need 3+ filters, have {len(good['filter'].unique())}: {sorted(good['filter'].unique())}")
        return None
    
    # Create color image
    maker = ColorImageMaker(
        wavelengths={'g': 473, 'r': 622, 'i': 763, 'z': 905, 'u': 355},
        scaling_method='asinh',
        verbose=verbose
    )
    
    try:
        # Rename obs_mjd to mjd for ColorImageMaker
        good_copy = good.copy()
        good_copy['mjd'] = good_copy['obs_mjd']

        rgb, info = maker.create_from_dataframe(
            good_copy,
            output_path=output_name,
            check_mjd_range=0.01  # Warn if >0.01 days apart
        )
        return rgb, maker, info
    except Exception as e:
        if verbose:
            print(f"Error creating color image: {e}")
        return None

### Create Color Images from Each Group

In [11]:
# Try to create color images from each temporal group
color_results = {}

for group_id, group in enumerate(temporal_groups):
    print(f"\n{'='*70}")
    print(f"Creating color composite for Group {group_id + 1}")
    print(f"{'='*70}")
    
    result = create_color_from_group(
        group,
        seeing_limit=20,
        output_name=f'/Users/kryanhinds/subphot_pipe/color_group_{group_id}.png',
        verbose=True
    )
    
    if result is not None:
        rgb, maker, info = result
        color_results[group_id] = {'rgb': rgb, 'maker': maker, 'info': info, 'group': group}
        print(f"✓ Successfully created color image")
    else:
        print(f"✗ Failed to create color image")


Creating color composite for Group 1
After seeing filter (<20"): 12 / 16 images
Creating color image from 12 files
Filters: ['g', 'i', 'r', 'z']
MJD range: 61115.974 - 61115.989
Instruments: ['LT/IOO']

Reading FITS files...
  ✓ h_e_20260316_40_1_1_1.fits: g (2056, 2048)
  ✓ h_e_20260316_40_2_1_1.fits: g (2056, 2048)
  ✓ h_e_20260316_40_3_1_1.fits: g (2056, 2048)
  ✓ h_e_20260316_41_1_1_1.fits: r (2056, 2048)
  ✓ h_e_20260316_41_2_1_1.fits: r (2056, 2048)
  ✓ h_e_20260316_41_3_1_1.fits: r (2056, 2048)
  ✓ h_e_20260316_42_1_1_1.fits: i (2056, 2048)
  ✓ h_e_20260316_42_2_1_1.fits: i (2056, 2048)
  ✓ h_e_20260316_42_3_1_1.fits: i (2056, 2048)
  ✓ h_e_20260316_43_1_1_1.fits: z (2056, 2048)
  ✓ h_e_20260316_43_2_1_1.fits: z (2056, 2048)
  ✓ h_e_20260316_43_3_1_1.fits: z (2056, 2048)

Loaded filters: ['g', 'r', 'i', 'z']

Filter -> RGB mapping:
  g ( 473 nm) -> B
  r ( 622 nm) -> G
Error creating color image: 'i'
✗ Failed to create color image

Creating color composite for Group 2
After see

In [ ]:
print(f"\nSuccessfully created {len(color_results)} color composite images")

## Step 5: Display Color Images

In [ ]:
# Display all color images
for group_id, result in sorted(color_results.items()):
    print(f"\n{'='*70}")
    print(f"Group {group_id}")
    print(f"{'='*70}")
    
    maker = result['maker']
    group = result['group']
    
    # Show group info
    print(f"Time span: {group['obs_mjd'].max() - group['obs_mjd'].min():.4f} days")
    print(f"Objects: {', '.join(group['object'].unique())}")
    good_seeing = group[group['seeing'] < 2.0]
    print(f"Good seeing images (<2\"): {len(good_seeing)} / {len(group)}")
    print(f"Images used in composite: {len(result['rgb'])}")
    
    # Display
    fig = maker.plot(
        figsize=(12, 10),
        title=f'Group {group_id}: NOT + LT Color Composite',
        show_info=True
    )
    plt.show()

## Step 6: Summary and Output

In [ ]:
print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(f"\nTotal images loaded: {len(df)}")
print(f"NOT/ALFOSC images: {len(df[df['instrument'] == 'NOT/ALFOSC'])}")
print(f"LT/IOO images: {len(df[df['instrument'] == 'LT/IOO'])}")
print(f"\nSeeing information:")
print(f"  Images with seeing: {df['seeing'].notna().sum()} / {len(df)}")
print(f"  LT seeing from L1SEESEC header: {len(df[(df['instrument'] == 'LT/IOO') & (df['seeing'].notna())])} images")
print(f"\nTemporal groups found: {len(temporal_groups)}")
print(f"Color composites created: {len(color_results)}")
print(f"\nOutput files saved to:")
for group_id in sorted(color_results.keys()):
    print(f"  /Users/kryanhinds/subphot_pipe/color_group_{group_id}.png")